# 05 — Generalization: the gap between train and test

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/005-generalization.ipynb)

A model that scores perfectly on the data it was fitted to may still be useless on new data. This notebook fits the same twelve noisy sine points with polynomials of growing degree, measures the error on rows it saw and rows it did not, and shows the moment the two numbers part company.

## 1. The numbers: twelve points, split in two

The target is a sine wave with a little noise on top. Four of the twelve rows are held out from fitting — the model never sees their targets — so the error on them measures something the training error cannot: behaviour on data that does not exist yet.

In [ ]:
import numpy as np                                        # the array library
rng = np.random.default_rng(21)                           # fixed seed: the same noise on every run
x = np.linspace(0, 1, 12)                                 # twelve inputs spread across the range
y = np.sin(2 * np.pi * x) + rng.normal(0, 0.12, 12)       # one full sine wave plus measurement noise
idx = np.arange(12)                                       # row numbers 0 to 11
train = idx % 3 != 0                                      # eight rows used for fitting
test = ~train                                             # four rows kept back for grading
print("x:", np.round(x, 2))                               # the inputs
print("y:", np.round(y, 2))                               # the noisy targets
print("train x:", np.round(x[train], 2))                  # where the model may look
print("test  x:", np.round(x[test], 2))                   # where it is graded
print("counts: train", train.sum(), " test", test.sum())   # eight and four

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import numpy as np</code> — loads NumPy.</li><li><code>rng = np.random.default_rng(21)</code> — a seeded random generator, so results are reproducible.</li><li><code>x = np.linspace(0, 1, 12)</code> — twelve inputs from 0 to 1.</li><li><code>y = ...</code> — the sine of <code>2*pi*x</code>, plus Gaussian noise with spread 0.12.</li><li><code>idx = np.arange(12)</code> — row labels, used only to build the split.</li><li><code>train = idx % 3 != 0</code> — a boolean mask: True for the eight fitting rows.</li><li><code>test = ~train</code> — the complement: the four held-out rows.</li><li><code>print("x:" ...)</code> — the twelve inputs.</li><li><code>print("y:" ...)</code> — the noisy targets.</li><li><code>print("train x" ...)</code> — where the model may look.</li><li><code>print("test x" ...)</code> — confirms the split is interleaved, not one block.</li><li><code>print("counts" ...)</code> — eight rows for fitting, four for grading.</li></ul>

</details>

## 2. Step by step: three models, two errors, one gap

`np.polyfit` finds the polynomial of a chosen degree with the smallest squared error on the training rows. Raise the degree and the model gains freedom: a degree-9 curve has ten coefficients for eight points, so it can pass almost exactly through every one of them.

In [ ]:
import warnings                                           # keep NumPy's conditioning warning out of the way
warnings.filterwarnings("ignore")                         # degree 9 on 8 points is barely determined

for deg in [1, 3, 9]:                                     # a line, then a cubic, then a wiggle machine
    coef = np.polyfit(x[train], y[train], deg)             # least squares on the training rows only
    pred_train = np.polyval(coef, x[train])                # predictions on rows it learned from
    pred_test = np.polyval(coef, x[test])                  # predictions on rows it never saw
    err_train = np.mean((pred_train - y[train]) ** 2)       # mean squared error, training
    err_test = np.mean((pred_test - y[test]) ** 2)          # mean squared error, held out
    print(f"degree {deg}: train {err_train:.4f}  test {err_test:.4f}  gap {err_test - err_train:+.4f}")   # the gap is the story

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import warnings</code> — NumPy warns when a fit is nearly undetermined.</li><li><code>warnings.filterwarnings("ignore")</code> — silence that warning so the numbers stay readable.</li><li><code>for deg in [1, 3, 9]</code> — three complexities, from a straight line to a nine-degree curve.</li><li><code>coef = np.polyfit(...)</code> — fits the polynomial by least squares on the training rows.</li><li><code>pred_train = np.polyval(...)</code> — evaluates that polynomial at the training inputs.</li><li><code>pred_test = np.polyval(...)</code> — evaluates it at the held-out inputs.</li><li><code>err_train = ...</code> — average squared miss on rows the fit has seen.</li><li><code>err_test = ...</code> — average squared miss on rows it has not.</li><li><code>print(f"degree {deg} ...")</code> — both errors and their difference, side by side.</li></ul>

</details>

## 3. Plot: error against complexity

Sweep the degree from 1 to 9 and plot both errors on a log scale. The training curve only ever falls — more freedom always fits the sample better. The test curve is U-shaped, and its minimum is the complexity worth choosing.

In [ ]:
import matplotlib.pyplot as plt                          # plotting library
degrees = np.arange(1, 10)                               # every degree from 1 to 9
train_err = []                                           # one training error per degree
test_err = []                                            # one test error per degree
for d in degrees:                                        # one iteration per model complexity
    coef = np.polyfit(x[train], y[train], d)              # refit at this degree
    train_err.append(np.mean((np.polyval(coef, x[train]) - y[train]) ** 2))   # seen rows
    test_err.append(np.mean((np.polyval(coef, x[test]) - y[test]) ** 2))      # held-out rows
plt.figure(figsize=(6, 3))                               # a wide, short figure
plt.plot(degrees, train_err, "o-", color="tab:blue", label="train")   # the falling curve
plt.plot(degrees, test_err, "o-", color="tab:red", label="test")      # the U-shaped curve
plt.yscale("log")                                        # the errors span several orders of magnitude
plt.xlabel("polynomial degree (model freedom)")          # x-axis label
plt.ylabel("mean squared error, log scale")              # y-axis label
plt.legend()                                             # name the two lines
plt.title("More freedom lowers train error and raises test error")   # the takeaway
plt.show()                                               # render

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import matplotlib.pyplot as plt</code> — the plotting library.</li><li><code>degrees = np.arange(1, 10)</code> — 1 through 9.</li><li><code>train_err = []</code> — one training error per degree.</li><li><code>test_err = []</code> — one test error per degree.</li><li><code>for d in degrees</code> — one model per degree.</li><li><code>coef = np.polyfit(...)</code> — the best polynomial of that degree on the training rows.</li><li><code>train_err.append(...)</code> — training error stored in order.</li><li><code>test_err.append(...)</code> — held-out error stored in order.</li><li><code>plt.figure(...)</code> — a wide, short figure.</li><li><code>plt.plot(degrees, train_err, "o-")</code> — markers plus a line, one point per degree.</li><li><code>plt.plot(degrees, test_err, "o-")</code> — the same for the held-out error.</li><li><code>plt.yscale("log")</code> — a log axis, because test error jumps by a factor of ten.</li><li><code>plt.xlabel(...)</code> — model freedom, from low to high.</li><li><code>plt.ylabel(...)</code> — error, on a log scale.</li><li><code>plt.legend()</code> — names the two curves.</li><li><code>plt.title(...)</code> — what is being traded against what.</li><li><code>plt.show()</code> — renders the figure.</li></ul>

</details>

## 4. Interactive: drag the degree

The slider refits the polynomial at each degree. Blue points were used for fitting, green points were not, and the grey line is the noise-free sine nobody gets to see. Watch the red curve stop following the wave and start following the noise.

In [ ]:
from ipywidgets import interact                            # sliders that rerun a function
fine = np.linspace(0, 1, 200)                             # a dense grid across the data range
truth = np.sin(2 * np.pi * fine)                          # the noise-free pattern the data came from

def show(deg=3):                                         # deg comes from the slider
    coef = np.polyfit(x[train], y[train], deg)            # fit at the chosen complexity
    tr = np.mean((np.polyval(coef, x[train]) - y[train]) ** 2)   # training error
    te = np.mean((np.polyval(coef, x[test]) - y[test]) ** 2)     # held-out error
    plt.figure(figsize=(6, 3))                           # a fresh figure for each drag
    plt.plot(fine, truth, color="grey", alpha=0.6, label="true sine")          # the hidden pattern
    plt.plot(fine, np.polyval(coef, fine), color="tab:red", label=f"degree {deg}")   # the fitted curve
    plt.scatter(x[train], y[train], color="tab:blue", label="train")            # rows used for fitting
    plt.scatter(x[test], y[test], color="tab:green", s=70, label="test")        # rows held back
    plt.ylim(-2.5, 3)                                    # fixed axes so frames do not jump
    plt.legend(loc="upper right", fontsize=8)            # name every element
    plt.title(f"degree {deg}: train {tr:.4f} | test {te:.4f}")   # live readout
    plt.show()                                           # draw the frame

interact(show, deg=(1, 9, 1))                            # attach the slider

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>from ipywidgets import interact</code> — the widget helper.</li><li><code>fine = np.linspace(-0.1, 1.1, 200)</code> — a dense grid, slightly wider than the data.</li><li><code>truth = np.sin(...)</code> — the noiseless sine, for reference only.</li><li><code>def show(deg=3)</code> — redraws one frame per slider value.</li><li><code>coef = np.polyfit(...)</code> — refits the model at that degree.</li><li><code>tr = ...</code> — training error, for the readout.</li><li><code>te = ...</code> — held-out error, for the readout.</li><li><code>plt.figure(...)</code> — a fresh figure per drag.</li><li><code>plt.plot(fine, truth, ...)</code> — the pattern the data was drawn from.</li><li><code>plt.plot(fine, np.polyval(coef, fine), ...)</code> — the model, evaluated on the dense grid.</li><li><code>plt.scatter(...)</code> — the blue training rows.</li><li><code>plt.scatter(...)</code> — the green held-out rows, drawn larger.</li><li><code>plt.ylim(-2.5, 3)</code> — clamps the view so high degrees do not blow up the axes.</li><li><code>plt.legend(...)</code> — names all four elements in a corner.</li><li><code>plt.title(f"...")</code> — live numbers, so you can read the gap as you drag.</li><li><code>plt.show()</code> — draws the frame.</li><li><code>interact(show, deg=(1, 9, 1))</code> — slider from degree 1 to 9, step 1.</li></ul>

</details>

## 5. When it breaks: degree 9 memorises the noise

At degree 9 the ten coefficients can satisfy eight training points almost exactly, so the training error collapses to zero — the model has learned the noise, not the wave. The four held-out points then land far from the curve, and the test error is the only number that warns you.

In [ ]:
coef9 = np.polyfit(x[train], y[train], 9)                # ten coefficients, eight points: almost exact
print("train error:", round(np.mean((np.polyval(coef9, x[train]) - y[train]) ** 2), 6))   # essentially zero
print("test  error:", round(np.mean((np.polyval(coef9, x[test]) - y[test]) ** 2), 4))      # many times larger
for j in np.where(test)[0]:                              # inspect every held-out row
    print(f"x={x[j]:.2f}  truth={y[j]:+.3f}  predicted={np.polyval(coef9, x[j]):+.3f}")   # where it misses
plt.figure(figsize=(6, 3))                               # a wide, short figure
plt.plot(fine, np.polyval(coef9, fine), color="tab:red", label="degree 9 fit")   # the memorising curve
plt.scatter(x[train], y[train], color="tab:blue", label="train")                 # points it was fitted to
plt.scatter(x[test], y[test], color="tab:green", s=70, label="test")             # points it must generalise to
plt.ylim(-2.5, 3)                                        # clip the extremes so the shape stays visible
plt.legend(loc="upper right", fontsize=8)                # name the elements
plt.title("Zero training error, large test error: the gap is overfitting")   # the takeaway
plt.show()                                               # render

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>coef9 = np.polyfit(...)</code> — one polynomial of degree 9, fitted on eight rows.</li><li><code>print("train error" ...)</code> — rounded to six places because it is nearly zero.</li><li><code>print("test error" ...)</code> — the same model on rows it never saw.</li><li><code>for j in np.where(test)[0]</code> — loops over the four held-out row numbers.</li><li><code>print(f"x={x[j]:.2f} ...")</code> — truth and prediction on each held-out row.</li><li><code>plt.figure(...)</code> — a wide, short figure.</li><li><code>plt.plot(fine, np.polyval(coef9, fine), ...)</code> — the fitted curve across the whole range.</li><li><code>plt.scatter(...)</code> — the blue training points it memorised.</li><li><code>plt.scatter(...)</code> — the larger green test points it must generalise to.</li><li><code>plt.ylim(-2.5, 3)</code> — keeps the interesting part in frame.</li><li><code>plt.legend(...)</code> — names the curve and both point sets.</li><li><code>plt.title(...)</code> — names the failure: good on train, bad on test.</li><li><code>plt.show()</code> — renders the figure.</li></ul>

</details>

## Your turn

Change the split so that `test` holds out the two end points (`idx == 0` or `idx == 11`) and rerun section 2 — the gap for degree 9 should get much larger. Then find the degree with the lowest test error in section 4, and say why you cannot use that number to choose a model in real life.